In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn openpyxl

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print("Customer Segmentation Environment Ready!")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
df = pd.read_csv("customer_segmentation.csv")

print("Dataset loaded successfully!")
print("Rows and Columns:", df.shape)

display(df.head())

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.isna().sum()

In [ ]:
df.isna().sum().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isna().sum().sum()

In [ ]:
df.describe()

In [ ]:
df['Education'].value_counts()

In [ ]:
df['Marital_Status'].value_counts()

DATA CLEANING AND FEATURE ENGINEERING

In [ ]:
df["Dt_Customer"]=pd.to_datetime(df["Dt_Customer"],dayfirst=True)

In [ ]:
df.info()

In [ ]:
# Calculate Age using Year_Birth (using 2025 as the baseline year)
df["Age"] = 2025 - df["Year_Birth"]

In [ ]:
df["Age"]

In [ ]:
df.head()

In [ ]:
df["Total_Children"] = df["Kidhome"] + df["Teenhome"]
df["Total_Children"]

In [ ]:
df.columns

In [ ]:
spend_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
df["Total_Spend"] = df[spend_cols].sum(axis=1)

In [ ]:
df["Total_Spend"]

In [ ]:
df["Customer_Since"] = (pd.Timestamp("today") - df["Dt_Customer"]).dt.days

In [ ]:
df["Customer_Since"]

In [ ]:
sns.histplot(df["Age"], bins=30, kde=True)
plt.title("Distribution of Customer Ages")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

In [ ]:
sns.histplot(df["Income"],bins=30,kde=True)
plt.title("Distribution of Customer Income")
plt.xlabel("Income")
plt.ylabel("Count")
plt.show()

In [ ]:
sns.histplot(df["Total_Spend"], bins=30, kde=True)
plt.title("Distribution of Customer Spending")
plt.xlabel("Spending")
plt.ylabel("Count")
plt.show()

In [ ]:
sns.boxplot(x="Education",y="Income", data=df)
plt.xticks(rotation=45)
plt.title("Income Distribution by Education Level")
plt.xlabel("Education")
plt.ylabel("Income")

In [ ]:
sns.boxplot(x="Marital_Status",y="Total_Spend", data=df)
plt.xticks(rotation=45)
plt.title("Spending by Marital Status")
plt.show()

In [ ]:
df.columns

In [ ]:
corr = df[["Income", "Age", "Recency", "Total_Spend", "NumWebPurchases", "NumStorePurchases"]].corr()
corr

In [ ]:
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
pivot_income = df.pivot_table(values="Income", index="Education", columns="Marital_Status", aggfunc="mean")
pivot_income

In [ ]:
sns.heatmap(pivot_income, annot=True, fmt="0f", cmap="YlGnBu")
plt.title("Average Income by Education and Marital Status")
plt.show()

In [ ]:
group1 = df.groupby("Education")["Total_Spend"].mean().sort_values(ascending=False)
group1

In [ ]:
group1.plot(kind="bar",
color="skyblue", title="Average Spending by Education Level")
plt.xlabel("Education")
plt.ylabel("Average Spending")
plt

In [ ]:
df["AcceptedAny"]=df[["AcceptedCmp1","AcceptedCmp2","AcceptedCmp3","AcceptedCmp4","AcceptedCmp5","Response"]].sum(axis=1)
df["AcceptedAny"].unique()

In [ ]:
df["AcceptedAny"]=df["AcceptedAny"].apply(lambda x: 1 if x>=1 else 0)
df["AcceptedAny"].unique()

In [ ]:
group2 = df.groupby("Marital_Status")["AcceptedAny"].mean().sort_values(ascending=False)
group2

In [ ]:
group2.plot(kind="bar", color="lightgreen", title="Acceptance Rate by Marital Status")
plt.xlabel("Marital Status")
plt.ylabel("Acceptance Rate")
plt.show()

In [ ]:
bins = [18, 30, 40, 50, 60, 70, 90]
labels = ["18-29", "30-39", "40-49", "50-59", "60-69", "70+"]

In [ ]:
df["AgeGroup"] = pd.cut(df["Age"], bins=bins, labels=labels)

In [ ]:
df["AgeGroup"]

In [ ]:
group3 = df.groupby("AgeGroup")["Income"].mean()
group3

In [ ]:
group3.plot(kind="barh", color="orange", title="Average Income by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Average Income")
plt.show()

In [ ]:
df.head()

In [ ]:
df.columns

"Age", "Income", "Total_Spend", "NumWebPurchases", "NumStorePurchases", "NumWebVisitsMonth", "Recency"]

In [ ]:
features = ["Age", "Income", "Total_Spend", "NumWebPurchases", "NumStorePurchases", "NumWebVisitsMonth", "Recency"]

In [ ]:
X = df[features].copy()

In [ ]:
X

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [ ]:
X_scaled = scaler.fit_transform(X)

In [ ]:
X_scaled

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
wcss =[]

In [ ]:
for i in range(2,10):
     kmeans = KMeans(n_clusters=i)
     kmeans.fit(X_scaled)
     wcss.append(kmeans.inertia_)

In [ ]:
wcss

In [ ]:
plt.plot(range(2,10),wcss,marker = "o")
plt.title("The Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.show()

In [ ]:
kmeans = KMeans(n_clusters =6)
df["Clusters"] = kmeans.fit_predict(X_scaled)

In [ ]:
df["Clusters"]

In [ ]:
df.head()

In [ ]:
cluster_summery = df.groupby("Clusters")[features].mean()

In [ ]:
cluster_summery

In [ ]:
df["Clusters"].value_counts()

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
pca_data = pca.fit_transform(X_scaled)
df["PCA1"], df["PCA2"] = pca_data[:, 0], pca_data[:, 1]

In [ ]:
pca_data

In [ ]:
df["PCA1"]

In [ ]:
df["PCA2"]

In [ ]:
sns.scatterplot(x="PCA1", y="PCA2", hue="Clusters", data=df, palette="Set1")
plt.title("Customer Segmentation using PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

In [ ]:
cluster_summery

In [ ]:
import joblib
joblib.dump(kmeans, "kmeans_model.pkl")
joblib.dump(scaler, "scaler.pkl")